# Ship Detection

This section will show how to detect ships in a raster dataset.

In [ ]:
# import geoai package
import geoai

# download sample data
raster_url = (
    "https://huggingface.co/datasets/giswqs/geospatial/resolve/main/ships_dubai.tif"
)
raster_path = geoai.download_file(raster_url, "ships_dubai.tif")

In [ ]:
# Move the downloaded files to the target folder

import os
import shutil
from pathlib import Path

# ==== INPUTS ====

target_folder = "data/geoai/ship-detection"  # Enter your target folder path here
datasets = ["ships_dubai.tif"]  # Add dataset names as a list
source_dir = Path(".")  # Current location of datasets

# ==== SCRIPT ====

target_path = Path(target_folder)
target_path.mkdir(parents=True, exist_ok=True)
for dataset_name in datasets:
    source_path = source_dir / dataset_name
    
    if not source_path.exists():
        print(f"Dataset '{dataset_name}' not found in {source_dir}")
        continue
    
    if source_path.is_file():
        files_to_move = [source_path]
    else:
        files_to_move = list(source_path.parent.glob(f"{dataset_name}.*"))
    
    for file_path in files_to_move:
        dest = target_path / file_path.name
        shutil.move(str(file_path), str(dest))
        print(f"Moved: {file_path.name} -> {target_folder}")
        
print("Done!")

In [ ]:
# Visualize the dataset

raster = "data/geoai/ship-detection/ships_dubai.tif"
geoai.view_raster(raster)

In [ ]:
# Initialize model

detector = geoai.ShipDetector()

In [ ]:
# Generate masks

output_path = "data/geoai/ship-detection/ships.geojson"
masks_path = detector.generate_masks(
    raster,
    output_path=output_path,
    confidence_threshold=0.9,
    mask_threshold=0.7,
    overlap=0.25,
    chip_size=(256, 256),
    batch_size=4,
)

In [ ]:
# Vectorize masks

gdf = detector.vectorize_masks(
    output_path,
    output_path="data/geoai/ship-detection/ships_dubai_masks.geojson",
    confidence_threshold=0.8,
    min_object_area=100,
    max_object_size=10000,
)

In [ ]:
# Visualize initial results

geoai.view_vector_interactive(gdf, column="confidence", tiles=raster_url)

In [ ]:
# Calculating geometric properties

gdf = geoai.add_geometric_properties(gdf)
gdf.head()

In [ ]:
geoai.view_vector_interactive(gdf, column="confidence", tiles=raster_url)

In [ ]:
# Filter results by area

m = geoai.view_raster(raster_url, backend="ipyleaflet")
m

In [ ]:
# Use the drawing tool to select the area of interest.

aoi = m.user_rois

if aoi is None:

    aoi = {
        "type": "FeatureCollection",
        "features": [
            {
                "type": "Feature",
                "properties": {},
                "geometry": {
                    "type": "Polygon",
                    "coordinates": [
                        [
                            [55.133729, 25.110277],
                            [55.134072, 25.11393],
                            [55.134823, 25.115601],
                            [55.136025, 25.117116],
                            [55.137677, 25.118127],
                            [55.140145, 25.118787],
                            [55.142248, 25.11902],
                            [55.142012, 25.118243],
                            [55.140831, 25.116728],
                            [55.13948, 25.116903],
                            [55.137956, 25.116825],
                            [55.136132, 25.115543],
                            [55.13566, 25.114416],
                            [55.135467, 25.1136],
                            [55.135939, 25.112609],
                            [55.136218, 25.111657],
                            [55.13551, 25.110685],
                            [55.134373, 25.110102],
                            [55.133729, 25.110277],
                        ]
                    ],
                },
            }
        ],
    }

In [ ]:
import geopandas as gpd

aoi_gdf = gpd.GeoDataFrame.from_features(aoi["features"], crs="EPSG:4326").to_crs(
    gdf.crs
)

In [ ]:
# Intersect the selected area with the vectorized masks to filter the results.

gdf_filter = gdf[gdf.intersects(aoi_gdf.geometry[0])]
gdf_filter.head()

In [ ]:
# Visualize final results

geoai.view_vector_interactive(gdf_filter, column="confidence", tiles=raster_url)

In [ ]:
# Save the final result

gdf_filter.to_file("data/geoai/ship-detection/ships_dubai.geojson")